# 04 Elasticity

Thin notebook over the phase script(s) in `scripts/`. Cells are split at the `# ----` section markers. Run from the repo root (`jupyter lab`), or just run the scripts.

In [ ]:
import os, sys
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.insert(0, os.getcwd())
%matplotlib inline

## scripts/30_phase3_elasticity.py

In [ ]:
"""Phase 3: own-price elasticity per product.

ln(units) = a + b * ln(shelf_price) + store FE + week_of_year FE + feature + display + e, fitted per product
(separate OLS per UPC, so b is the product's own elasticity). Standard errors clustered by store.
A second specification WITHOUT the promo flags is reported to show how much of the 'price effect' is really
display/feature support riding on the price variable.
Outputs: outputs/phase3_elasticity.csv, outputs/phase3_results.md, figure.
"""
import json
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")  # rank-deficiency warnings for store dummies with few rows

import matplotlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

ROOT = Path(os.getcwd())
sys.path.insert(0, str(ROOT))
from src.load import build_clean  # noqa: E402

FIG = ROOT / "outputs" / "figures"
OUT = ROOT / "outputs" / "phase3_results.md"
lines = []; P = lines.append

df = build_clean()
df = df[(df.units > 0) & (df.price > 0)].copy()
df["ln_units"] = np.log(df.units); df["ln_price"] = np.log(df.price)

rows = []
for upc, g in df.groupby("upc"):
    if g.store_num.nunique() < 5 or len(g) < 500 or g.ln_price.std() < 0.02:
        continue
    base = {"upc": upc, "description": g.description.iloc[0], "category": g.category.iloc[0], "n": len(g),
            "n_stores": g.store_num.nunique(), "price_cv": float(g.price.std() / g.price.mean())}
    try:
        m_full = smf.ols("ln_units ~ ln_price + feature + display + C(store_num) + C(week_of_year)", data=g).fit(
            cov_type="cluster", cov_kwds={"groups": g.store_num})
        m_nop = smf.ols("ln_units ~ ln_price + C(store_num) + C(week_of_year)", data=g).fit(
            cov_type="cluster", cov_kwds={"groups": g.store_num})
    except Exception as e:  # singular designs etc.
        base["error"] = str(e)[:80]; rows.append(base); continue
    base.update({"elasticity": m_full.params["ln_price"], "se": m_full.bse["ln_price"],
                 "ci_lo": m_full.conf_int().loc["ln_price", 0], "ci_hi": m_full.conf_int().loc["ln_price", 1],
                 "feature_effect": m_full.params["feature"], "display_effect": m_full.params["display"],
                 "elasticity_no_promo_controls": m_nop.params["ln_price"], "r2": m_full.rsquared})
    rows.append(base)
res = pd.DataFrame(rows)
res.to_csv(ROOT / "outputs" / "phase3_elasticity.csv", index=False)

P("# Phase 3 results: own-price elasticity (generated by scripts/30_phase3_elasticity.py)\n")
P("Specification per product: `ln(units) ~ ln(shelf price) + feature + display + store FE + week-of-year FE`, "
  "OLS, standard errors clustered by store. Elasticity = coefficient on ln(price).\n")
ok = res.dropna(subset=["elasticity"])
P(f"- {len(ok)} products estimated. Median elasticity **{ok.elasticity.median():.2f}**, IQR {ok.elasticity.quantile(.25):.2f} to {ok.elasticity.quantile(.75):.2f}.")
P(f"- {int((ok.elasticity < 0).sum())} of {len(ok)} are negative; {int(ok.elasticity.between(-4, -1).sum())} fall in the textbook -1 to -4 band.")
P(f"- Without the feature/display controls the median elasticity is **{ok.elasticity_no_promo_controls.median():.2f}**: "
  "the price variable then absorbs display and circular support, overstating price sensitivity.")
P(f"- Mean effect of a feature on ln(units) holding price fixed: {ok.feature_effect.mean():+.2f} (x{np.exp(ok.feature_effect.mean()):.2f}); "
  f"of a display: {ok.display_effect.mean():+.2f} (x{np.exp(ok.display_effect.mean()):.2f}).")
cat = ok.groupby("category").elasticity.agg(["count", "median", "mean", "min", "max"]).reset_index()
P("\n## By category\n"); P(cat.to_markdown(index=False, floatfmt=".2f"))
show = ok[["upc", "description", "category", "n", "elasticity", "ci_lo", "ci_hi", "elasticity_no_promo_controls", "feature_effect", "display_effect"]].sort_values("elasticity")
show["upc"] = show["upc"].astype(str)
P("\n## By product\n"); P(show.to_markdown(index=False, floatfmt=".2f"))
P("\n## Assumption stated honestly\n")
P("Price is treated as exogenous conditional on store, seasonality and promo-support controls. It is not fully: retailers cut "
  "price when they expect demand to respond, and base-price changes may follow cost changes that also shift demand. "
  "With more data I would instrument shelf price with wholesale-cost shocks or use competitor pricing as a control. "
  "Treat these as reduced-form 'promotional' elasticities, which is what a trade-promotion planner needs anyway.")

fig, ax = plt.subplots(figsize=(8, 9))
s = show.sort_values("elasticity")
y = np.arange(len(s))
colors = s.category.map(dict(zip(sorted(s.category.unique()), ["#1f6f8b", "#c0392b", "#27ae60", "#8e44ad"])))
ax.errorbar(s.elasticity, y, xerr=[s.elasticity - s.ci_lo, s.ci_hi - s.elasticity], fmt="o", ecolor="grey", markersize=4, color="none")
ax.scatter(s.elasticity, y, c=colors, s=25, zorder=3)
ax.set_yticks(y); ax.set_yticklabels(s.description, fontsize=7)
ax.axvline(0, color="k", lw=.8); ax.axvspan(-4, -1, alpha=.08, color="green", label="typical FMCG band (-4 to -1)")
ax.set_xlabel("own-price elasticity (95% CI, clustered by store)"); ax.set_title("Own-price elasticity by product"); ax.legend(loc="lower right"); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(FIG / "phase3_elasticity.png", dpi=130); plt.close(fig)

json.dump({"median_elasticity": float(ok.elasticity.median()), "n_products": int(len(ok)), "n_negative": int((ok.elasticity < 0).sum()),
           "n_in_band": int(ok.elasticity.between(-4, -1).sum()), "median_no_controls": float(ok.elasticity_no_promo_controls.median()),
           "by_category": cat.to_dict(orient="records")}, open(ROOT / "outputs" / "phase3_summary.json", "w"), indent=2)
OUT.write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))